<a href="https://colab.research.google.com/github/ashok9847/ad-channel-intelligence-multi-channel-marketing-analytics/blob/main/ad_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import duckdb

In [2]:
ad_spend =pd.read_csv('/content/drive/MyDrive/data-set/ad_spend.csv')
customers=pd.read_csv('/content/drive/MyDrive/data-set/customers.csv')
orders=pd.read_csv('/content/drive/MyDrive/data-set/orders.csv')



In [3]:
# Basic inspection - do this for all three
for name, df in [('ad_spend', ad_spend), ('customers', customers), ('orders', orders)]:
    print(f"--- {name} ---")
    print(df.shape)
    print(df.dtypes)
    print(df.isnull().sum())
    print(df.duplicated().sum(), "duplicate rows")
    print()

--- ad_spend ---
(1460, 3)
spend_date     object
channel        object
spend         float64
dtype: object
spend_date    0
channel       0
spend         0
dtype: int64
0 duplicate rows

--- customers ---
(1000, 3)
customer_id            object
first_order_date       object
acquisition_channel    object
dtype: object
customer_id            0
first_order_date       0
acquisition_channel    0
dtype: int64
0 duplicate rows

--- orders ---
(2200, 4)
order_id        object
customer_id     object
order_date      object
revenue        float64
dtype: object
order_id       0
customer_id    0
order_date     0
revenue        0
dtype: int64
0 duplicate rows



Here i am changing the data column in to the actual date and time


In [4]:
tables = [ad_spend, orders, customers]

for df in tables:
    for col in df.columns:
        if "date" in col.lower():
            df[col] = pd.to_datetime(df[col])
            print(df.dtypes)


spend_date    datetime64[ns]
channel               object
spend                float64
dtype: object
order_id               object
customer_id            object
order_date     datetime64[ns]
revenue               float64
dtype: object
customer_id                    object
first_order_date       datetime64[ns]
acquisition_channel            object
dtype: object


That means there are all the cutomers who has placed the order

In [5]:
orphan_orders = set(orders['customer_id']) - set(customers['customer_id'])
print("Orders with unknown customer:", len(orphan_orders))

# Every customer should have at least one order (or flag if not)
customers_no_orders = set(customers['customer_id']) - set(orders['customer_id'])
print("Customers with zero orders:", len(customers_no_orders))

# Check for duplicate order_ids, customer_ids
print(orders['order_id'].duplicated().sum())
print(customers['customer_id'].duplicated().sum())

Orders with unknown customer: 0
Customers with zero orders: 0
0
0


In [6]:
# we can see the row of the order and customer table is different i.e order table has more rows so this means that same customer has order more then one time which is valid
print(orders['customer_id'].duplicated().sum())

1200


# Now we have to join the tables with customer table with order table


In [7]:
# Step A: aggregate orders to customer level
customer_revenue = orders.groupby('customer_id').agg(
    total_revenue=('revenue', 'sum'),
    order_count=('order_id', 'count'),
    first_order=('order_date', 'min'),
    last_order=('order_date', 'max')
).reset_index()

# Step B: left join onto customers (left join keeps all customers, even those with 0 orders)
customer_full = customers.merge(customer_revenue, on='customer_id', how='left')

# Step C: any customer with no matching orders will show NaN — fill explicitly
customer_full['total_revenue'] = customer_full['total_revenue'].fillna(0)
customer_full['order_count'] = customer_full['order_count'].fillna(0)

print(customer_full.shape)
print(customer_full.head())

(1000, 7)
  customer_id first_order_date acquisition_channel  total_revenue  \
0   CUST00001       2025-04-13            Meta Ads         284.27   
1   CUST00002       2025-12-15          Google Ads         356.68   
2   CUST00003       2025-09-28          Google Ads         180.56   
3   CUST00004       2025-04-17          Google Ads         460.61   
4   CUST00005       2025-03-13          Google Ads         266.00   

   order_count first_order last_order  
0            2  2025-04-13 2025-05-03  
1            3  2025-12-15 2025-12-31  
2            2  2025-09-28 2025-11-28  
3            5  2025-04-17 2025-09-02  
4            1  2025-03-13 2025-03-13  


In [8]:
customer_full

,customer_id,first_order_date,acquisition_channel,total_revenue,order_count,first_order,last_order
0,CUST00001,2025-04-13,Meta Ads,284.27,2,2025-04-13,2025-05-03
1,CUST00002,2025-12-15,Google Ads,356.68,3,2025-12-15,2025-12-31
2,CUST00003,2025-09-28,Google Ads,180.56,2,2025-09-28,2025-11-28
3,CUST00004,2025-04-17,Google Ads,460.61,5,2025-04-17,2025-09-02
4,CUST00005,2025-03-13,Google Ads,266.00,1,2025-03-13,2025-03-13
...,...,...,...,...,...,...,...
995,CUST00996,2025-02-07,Google Ads,256.19,2,2025-02-07,2025-07-19
996,CUST00997,2025-10-20,Google Ads,393.26,3,2025-10-20,2025-12-23
997,CUST00998,2025-05-19,Google Ads,185.48,2,2025-05-19,2025-09-12
998,CUST00999,2025-12-22,Email,73.16,2,2025-12-22,2025-12-31


In [9]:
customer_full['total_revenue'].min() < 0#checking whether the min value is less then 0

False

In [10]:
customer_full.isnull().sum()# so it is confirm that no order has null value i.e every cuwstomer has placed an order

,0
customer_id,0
first_order_date,0
acquisition_channel,0
total_revenue,0
order_count,0
first_order,0
last_order,0


In [11]:
import duckdb
#spend by channel
spend_by_channel = duckdb.sql("""
SELECT
    channel,
    SUM(spend) AS total_spend
FROM ad_spend
GROUP BY channel
""").df()
spend_by_channel

,channel,total_spend
0,Google Ads,94186.32
1,Meta Ads,67961.01
2,Email,13160.19
3,TikTok Ads,33978.00


Aggregate ad spend by channel (and by month, for trend later)

In [12]:
spend_by_channel_month = duckdb.sql("""
SELECT
    channel,
    strftime(spend_date, '%Y-%m') AS month,
    SUM(spend) AS total_spend
FROM ad_spend
GROUP BY channel, month
ORDER BY month
""").df()
spend_by_channel_month

,channel,month,total_spend
0,Meta Ads,2025-01,5613.65
1,Google Ads,2025-01,7468.88
2,Email,2025-01,1088.72
3,TikTok Ads,2025-01,2813.50
4,Meta Ads,2025-02,5103.94
5,Email,2025-02,956.46
6,TikTok Ads,2025-02,2558.15
7,Google Ads,2025-02,6983.25
8,Meta Ads,2025-03,5555.81
9,Email,2025-03,1096.86


In [13]:
customers_per_channel= duckdb.sql("""
SELECT
    acquisition_channel,
    COUNT(customer_id) AS new_customers,
    round(SUM(total_revenue),0) AS total_revenue,
    SUM(order_count) AS total_orders
FROM customer_full
GROUP BY acquisition_channel
""").df()
customers_per_channel

,acquisition_channel,new_customers,total_revenue,total_orders
0,Meta Ads,352,96648.0,757.0
1,Google Ads,395,105269.0,858.0
2,Email,100,28451.0,237.0
3,TikTok Ads,153,40120.0,348.0


Join spend + customer/revenue data to compute CAC, LTV, ROAS

In [14]:
channel_summary = spend_by_channel.merge(
    customers_per_channel,
    left_on='channel', right_on='acquisition_channel',
    how='inner'
).drop(columns='acquisition_channel')

channel_summary['CAC'] = channel_summary['total_spend'] / channel_summary['new_customers']
channel_summary['LTV'] = channel_summary['total_revenue'] / channel_summary['new_customers']
channel_summary['LTV_to_CAC'] = channel_summary['LTV'] / channel_summary['CAC']
channel_summary['avg_orders_per_customer'] = channel_summary['total_orders'] / channel_summary['new_customers']
channel_summary['ROAS'] = channel_summary['total_revenue'] / channel_summary['total_spend']

print(channel_summary.sort_values('LTV_to_CAC', ascending=False))

      channel  total_spend  new_customers  total_revenue  total_orders  \
2       Email     13160.19            100        28451.0         237.0   
1    Meta Ads     67961.01            352        96648.0         757.0   
3  TikTok Ads     33978.00            153        40120.0         348.0   
0  Google Ads     94186.32            395       105269.0         858.0   

          CAC         LTV  LTV_to_CAC  avg_orders_per_customer      ROAS  
2  131.601900  284.510000    2.161899                 2.370000  2.161899  
1  193.071051  274.568182    1.422110                 2.150568  1.422110  
3  222.078431  262.222222    1.180764                 2.274510  1.180764  
0  238.446380  266.503797    1.117668                 2.172152  1.117668  


Monthly trend per channel (CAC and revenue over time)

In [19]:
customer_full['acq_month'] = customer_full['first_order_date'].dt.to_period('M').astype(str)

monthly_customers = customer_full.groupby(
    ['acquisition_channel', 'acq_month']
).size().reset_index(name='new_customers')

monthly_trend = spend_by_channel_month.merge(
    monthly_customers,
    left_on=['channel', 'month'],
    right_on=['acquisition_channel', 'acq_month'],
    how='left'
)

monthly_trend['CAC'] = (
    monthly_trend['total_spend'] / monthly_trend['new_customers']
)
print(channel_summary.sort_values('LTV_to_CAC', ascending=False))

      channel  total_spend  new_customers  total_revenue  total_orders  \
2       Email     13160.19            100        28451.0         237.0   
1    Meta Ads     67961.01            352        96648.0         757.0   
3  TikTok Ads     33978.00            153        40120.0         348.0   
0  Google Ads     94186.32            395       105269.0         858.0   

          CAC         LTV  LTV_to_CAC  avg_orders_per_customer      ROAS  
2  131.601900  284.510000    2.161899                 2.370000  2.161899  
1  193.071051  274.568182    1.422110                 2.150568  1.422110  
3  222.078431  262.222222    1.180764                 2.274510  1.180764  
0  238.446380  266.503797    1.117668                 2.172152  1.117668  


In [18]:
print(spend_by_channel_month["month"].dtype)
print(monthly_customers["acq_month"].dtype)

object
object


Repeat purchase behavior by channel

In [21]:
repeat_analysis = customer_full.groupby('acquisition_channel').agg(
    pct_repeat_customers=('order_count', lambda x: (x > 1).mean() * 100),
    avg_order_count=('order_count', 'mean')
).reset_index()

## Sanity-check your numbers before trusting them

In [23]:
# Total spend should match across your two spend aggregations
assert abs(ad_spend['spend'].sum() - spend_by_channel['total_spend'].sum()) < 0.01

# Total revenue should match
assert abs(orders['revenue'].sum() - customer_full['total_revenue'].sum()) < 0.01

print("Overall CAC:", ad_spend['spend'].sum() / len(customers))
print("Overall LTV:", orders['revenue'].sum() / len(customers))

Overall CAC: 209.28552
Overall LTV: 270.488


# Export cleaned/aggregated tables for visualization

In [29]:
# Save to disk (don't capture the return value, it's None)
channel_summary.to_csv('channel_summary.csv', index=False)
monthly_trend.to_csv('monthly_trend.csv', index=False)
customer_full.to_csv('customer_full_cleaned.csv', index=False)

# Now read them back fresh to verify
cs = pd.read_csv('channel_summary.csv')
mt = pd.read_csv('monthly_trend.csv')
cf = pd.read_csv('customer_full_cleaned.csv')

print(cs)
print(mt.isnull().sum())
print(cf.shape)

      channel  total_spend  new_customers  total_revenue  total_orders  \
0  Google Ads     94186.32            395       105269.0         858.0   
1    Meta Ads     67961.01            352        96648.0         757.0   
2       Email     13160.19            100        28451.0         237.0   
3  TikTok Ads     33978.00            153        40120.0         348.0   

          CAC         LTV  LTV_to_CAC  avg_orders_per_customer      ROAS  
0  238.446380  266.503797    1.117668                 2.172152  1.117668  
1  193.071051  274.568182    1.422110                 2.150568  1.422110  
2  131.601900  284.510000    2.161899                 2.370000  2.161899  
3  222.078431  262.222222    1.180764                 2.274510  1.180764  
channel                0
month                  0
total_spend            0
acquisition_channel    0
acq_month              0
new_customers          0
CAC                    0
dtype: int64
(1000, 8)
